# 🎯 Customer Segmentation — Exploratory Data Analysis
**Project:** E-Commerce Customer Segmentation
**Dataset:** 50,000 customers × 25 features


In [ ]:
import sys, os
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 100
print('Libraries loaded ✓')

## 1. Data Loading & Overview

In [ ]:
df = pd.read_csv('../data/ecommerce_customer_churn_dataset.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

## 2. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
missing_df['Missing %'].plot(kind='bar', ax=ax, color='coral')
ax.set_title('Missing Value Percentage by Column', fontsize=14, fontweight='bold')
ax.set_ylabel('Missing %')
ax.axhline(5, color='red', linestyle='--', label='5% threshold')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
print(missing_df.to_string())

## 3. Customer Demographics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Customer Demographics', fontsize=16, fontweight='bold')

# Age distribution
df['Age'].dropna().clip(5, 80).hist(bins=30, ax=axes[0,0], color='steelblue', edgecolor='white')
axes[0,0].set_title('Age Distribution'); axes[0,0].set_xlabel('Age')

# Gender distribution
df['Gender'].value_counts().plot(kind='pie', ax=axes[0,1], autopct='%1.1f%%', colors=['#4A90D9','#E07B54'])
axes[0,1].set_title('Gender Distribution'); axes[0,1].set_ylabel('')

# Top countries
df['Country'].value_counts().head(8).plot(kind='barh', ax=axes[1,0], color='teal')
axes[1,0].set_title('Top 8 Countries'); axes[1,0].set_xlabel('Customers')

# Membership years
df['Membership_Years'].hist(bins=20, ax=axes[1,1], color='orchid', edgecolor='white')
axes[1,1].set_title('Membership Years'); axes[1,1].set_xlabel('Years')

plt.tight_layout()
plt.show()

## 4. Spending Distributions

In [ ]:
spend_cols = ['Lifetime_Value', 'Average_Order_Value', 'Total_Purchases', 'Credit_Balance']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Spending Distributions', fontsize=16, fontweight='bold')
for ax, col in zip(axes.flatten(), spend_cols):
    df[col].dropna().clip(upper=df[col].quantile(0.99)).hist(bins=40, ax=ax, color='darkorange', edgecolor='white')
    ax.set_title(col.replace('_',' '))
    ax.axvline(df[col].median(), color='red', linestyle='--', label=f'Median={df[col].median():.0f}')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap

In [ ]:
num_df = df.select_dtypes(include=[np.number]).dropna(thresh=len(df)*0.7, axis=1)
corr = num_df.corr()
fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0,
            linewidths=0.3, ax=ax, cbar_kws={'shrink':0.8})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. RFM Analysis

In [ ]:
rfm = df[['Days_Since_Last_Purchase','Total_Purchases','Lifetime_Value']].dropna()
rfm.columns = ['Recency','Frequency','Monetary']

# Bin into quartiles
for col in rfm.columns:
    rfm[col+'_Score'] = pd.qcut(rfm[col], 4, labels=[1,2,3,4], duplicates='drop').astype(int)
rfm['Recency_Score'] = 5 - rfm['Recency_Score']  # lower recency = better
rfm['RFM_Total'] = rfm['Recency_Score'] + rfm['Frequency_Score'] + rfm['Monetary_Score']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('RFM Distribution', fontsize=14, fontweight='bold')
for ax, col, color in zip(axes, ['Recency','Frequency','Monetary'], ['steelblue','darkorange','green']):
    rfm[col].clip(upper=rfm[col].quantile(0.99)).hist(bins=40, ax=ax, color=color, edgecolor='white')
    ax.set_title(col); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('RFM Summary:')
print(rfm[['Recency','Frequency','Monetary']].describe().round(2).to_string())

## 7. Cluster Visualization (Post-Training)

In [ ]:
from pathlib import Path
labeled_path = '../data/labeled_data.csv'
if Path(labeled_path).exists():
    labeled = pd.read_csv(labeled_path)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Segment counts
    seg_counts = labeled['Segment'].value_counts()
    seg_counts.plot(kind='bar', ax=axes[0], color=plt.cm.tab10.colors[:len(seg_counts)])
    axes[0].set_title('Customers per Segment', fontweight='bold')
    axes[0].set_xlabel('Segment'); axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=20)
    
    # LTV by segment
    labeled.boxplot(column='Lifetime_Value', by='Segment', ax=axes[1])
    axes[1].set_title('Lifetime Value by Segment', fontweight='bold')
    axes[1].set_xlabel('Segment'); axes[1].set_ylabel('Lifetime Value')
    plt.suptitle('')
    plt.tight_layout()
    plt.show()
    print('Segment Summary:')
    print(labeled.groupby('Segment')['Lifetime_Value'].describe().round(2).to_string())
else:
    print('Run train.py first to generate labeled data.')

## 8. Load & Display PCA Visualization

In [ ]:
from IPython.display import Image
pca_path = '../artifacts/pca_visualization.png'
if Path(pca_path).exists():
    display(Image(filename=pca_path))
else:
    print('Run train.py first to generate the PCA plot.')